# Annotate BANC Neurite Bundles

Query BANC tables (`codex_annotations`, `cell_info`) to find neurons of a specific hemilineage in a specific neuropil/side, then generate Neuroglancer states for bundle annotation.

In [4]:
from caveclient import CAVEclient
import pandas as pd
import numpy as np
import sys


#sys.path.insert(0, 'C:\\Users\\tony\\Code\\connectomics')
#sys.path.insert(0, 'C:\\Users\\tony\\Code\\connectomics\\connectomics')
#import annot8or
#from annot8_utils import split_pt_pos_str_lists
from nglui.statebuilder import ViewerState
import nglui.parser as parser

In [5]:
client = CAVEclient()
client.info.get_datastacks()

['minnie35_public_v0',
 'fanc_sandbox',
 'flywire_fafb_sandbox',
 'fanc_production_mar2021',
 'mrgd',
 'pinky_sandbox',
 'minnie65_sandbox',
 'flywire_fafb_public',
 'minnie65_public']

In [ ]:
# Jelly First time use: token to access BANC:
##save token in users cloudC:\Users\JHS\.cloudvolume\secrets\
##cave-secret.json
##in here: 
#{
 # "token": "PASTE_YOUR_TOKEN_HERE"
#}
from caveclient import CAVEclient
client = CAVEclient("brain_and_nerve_cord")
client.info.get_datastack_info()


HTTPError: 403 Client Error: FORBIDDEN for url: https://global.daf-apis.com/info/api/v2/datastack/full/brain_and_nerve_cord content: b'{\n  "data": {\n    "auth_dataset": "BANC",\n    "required_permission": "view"\n  },\n  "error": "missing_permission",\n  "message": "Missing permission: view for dataset BANC"\n}\n'

In [6]:
dataset = 'brain_and_nerve_cord'
client = CAVEclient(dataset)
client.annotation.get_tables()

HTTPError: 403 Client Error: FORBIDDEN for url: https://global.daf-apis.com/info/api/v2/datastack/full/brain_and_nerve_cord content: b'{\n  "data": {\n    "auth_dataset": "BANC",\n    "required_permission": "view"\n  },\n  "error": "missing_permission",\n  "message": "Missing permission: view for dataset BANC"\n}\n'

## Explore codex_annotations and cell_info schemas
Run these cells once to understand what columns and values are available for hemilineage queries.

### codex_annotations

In [33]:
# Inspect codex_annotations schema and sample data
codex_meta = client.annotation.get_table_metadata('codex_annotations')
print('codex_annotations metadata:')
print(codex_meta)
print()

codex_df = client.materialize.query_table('codex_annotations', limit=50)
print('Columns:', codex_df.columns.tolist())
print('Shape:', codex_df.shape)
codex_df.head(10)

codex_annotations metadata:
{'id': 54, 'schema_type': 'cell_type_reference', 'table_name': 'codex_annotations', 'valid': True, 'created': '2025-07-16T19:37:03.061831', 'deleted': None, 'user_id': '386', 'description': "Testing of new annotations [Note: This table 'codex_annotations' will update the 'target_id' foreign_key when updates are made to the 'cell_representative_point' table] ", 'notice_text': None, 'reference_table': 'cell_representative_point', 'flat_segmentation_source': None, 'write_permission': 'PRIVATE', 'read_permission': 'PUBLIC', 'last_modified': '2025-07-16T19:37:03.061831', 'voxel_resolution': [1.0, 1.0, 1.0]}



Columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'pt_supervoxel_id', 'pt_root_id', 'pt_position']
Shape: (50, 12)


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,0,2026-02-02 00:00:00+00:00,True,0,region,neck_connective,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
1,114361,2026-02-02 00:00:00+00:00,True,0,side,right,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
2,227822,2026-02-02 00:00:00+00:00,True,0,flow,intrinsic,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
3,316933,2026-02-02 00:00:00+00:00,True,0,super_class,descending,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
4,405961,2026-02-02 00:00:00+00:00,True,0,cell_class,descending_neuron,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
5,501177,2026-02-02 00:00:00+00:00,True,0,cell_type,DNp32,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
6,653324,2026-02-02 00:00:00+00:00,True,0,hemilineage,putative_primary,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
7,699124,2026-02-02 00:00:00+00:00,True,0,sexually_dimorphic,isomorphic,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
8,810039,2026-02-02 00:00:00+00:00,True,0,neuropeptide_verified,Ms,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
9,815967,2026-02-02 00:00:00+00:00,True,0,neurotransmitter_predicted,serotonin,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"


In [34]:
codex_df.cell_type.unique()

<StringArray>
[      'neck_connective',                 'right',             'intrinsic',
            'descending',     'descending_neuron',                 'DNp32',
      'putative_primary',            'isomorphic',                    'Ms',
             'serotonin', 'id:720575940630610042',              'id:11477',
         'id:5813050455',                 'id:60',                'id:355',
                'id:847',                'DNg105',                   'MX3',
                  'gaba', 'id:720575940647029044',              'id:10086',
                  'left',             'ascending',      'ascending_neuron',
              'AN05B006',                   '05B',           'AN_multi_87',
 'id:720575940630609170',              'id:10758']
Length: 29, dtype: string

In [35]:
codex_df.classification_system.unique()

<StringArray>
[                    'region',                       'side',
                       'flow',                'super_class',
                 'cell_class',                  'cell_type',
                'hemilineage',         'sexually_dimorphic',
      'neuropeptide_verified', 'neurotransmitter_predicted',
         'fafb_783_cell_type',         'manc_121_cell_type',
    'hemibrain_121_cell_type',          'fafb_783_match_id',
          'manc_121_match_id',     'hemibrain_121_match_id',
                    'user_id',  'neurotransmitter_verified']
Length: 18, dtype: string

### cell_info

In [36]:
# Inspect cell_info schema and sample data
cell_info_meta = client.annotation.get_table_metadata('cell_info')
print('cell_info metadata:')
print(cell_info_meta)
print()

cell_info_df = client.materialize.query_table('cell_info', limit=2000)
print('Columns:', cell_info_df.columns.tolist())
print('Shape:', cell_info_df.shape)
cell_info_df.head(10)

cell_info metadata:
{'id': 4, 'schema_type': 'bound_double_tag_user', 'table_name': 'cell_info__wclee_fly_cns_001', 'valid': True, 'created': '2023-10-30T00:00:01.188701', 'deleted': None, 'user_id': '4741', 'description': 'A general-purpose cell type / cell information table. Included are cell types (e.g. broad types like motor neuron, central neuron, sensory neuron, & glia, plus more specific subtypes of each of those), anatomical descriptions (e.g. ascending, descending, soma in brain, soma in VNC), and specific neuron identities (e.g. giant fiber, DNa01), and more – see https://banc.community/Annotations-(cell-types,-etc.) for a list of all annotations that can be found here. \n\nEach row of this table is a key-value pair, with the "tag2" column being the key (or parent node in the annotation tree) and the "tag" column being the actual annotation (or child node in the annotation tree). Example key-value pairs are "soma region"-"soma in VNC", "primary class"-"glia", "primary class"-

Columns: ['id', 'created', 'superceded_id', 'valid', 'tag', 'tag2', 'user_id', 'pt_supervoxel_id', 'pt_root_id', 'pt_position']
Shape: (2000, 10)


,id,created,superceded_id,valid,tag,tag2,user_id,pt_supervoxel_id,pt_root_id,pt_position
0,54,2023-11-11 17:35:32.081991+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,76071705571238495,720575941652633585,"[116797, 92500, 2956]"
1,55,2023-11-12 14:36:34.958330+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,76001336827081255,720575941545312005,"[115253, 92500, 2959]"
2,56,2023-11-12 16:36:38.929404+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,75930968083101983,720575941390623382,"[113199, 92500, 3040]"
3,58,2023-11-12 18:27:19.632643+00:00,<NA>,True,ascending,anterior-posterior projection pattern,2660,75930968082569810,720575941435998094,"[113276, 92500, 2847]"
4,59,2023-11-12 18:28:14.602559+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,75930968082496232,720575941626175643,"[113218, 92500, 2823]"
5,60,2023-11-12 18:40:10.759619+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,75930968082877277,720575941526955266,"[113827, 92500, 2961]"
6,61,2023-11-13 00:57:41.702534+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,75930968082675467,720575941508047639,"[113634, 92500, 2883]"
7,62,2023-11-13 16:01:53.117437+00:00,<NA>,True,ascending,anterior-posterior projection pattern,2660,76001336692337941,720575941449745333,"[116227, 92500, 2538]"
8,63,2023-11-13 16:02:51.785790+00:00,<NA>,True,ascending,anterior-posterior projection pattern,2660,76071705504320138,720575941627244396,"[118479, 92500, 2772]"
9,64,2023-11-13 16:04:02.741154+00:00,<NA>,True,descending,anterior-posterior projection pattern,2660,76071705504203138,720575941561655341,"[117318, 92500, 2738]"


In [37]:
cell_info_df.tag.unique()

<StringArray>
[            'descending',              'ascending',         'sensory neuron',
             'CNS neuron',             'unilateral',                  'DNx01',
            'giant fiber',             'soma in T2',             'soma in T1',
 'T1 “multinerve” neuron',      'hair plate neuron',           'soma on left',
          'soma on right']
Length: 13, dtype: string

In [38]:
# Check which columns contain hemilineage-like data
# Adjust column names below after inspecting the output above
for col in codex_df.columns:
    try:
        unique_vals = codex_df[col].nunique()
    except TypeError:
        unique_vals = codex_df[col].apply(tuple).nunique()
    print(f'{col}: {unique_vals} unique values')
    if unique_vals < 30:
        try:
            print(f'  Values: {codex_df[col].unique()}')
        except TypeError:
            print(f'  Values: {codex_df[col].apply(tuple).unique()}')

id: 50 unique values
created: 1 unique values
  Values: <DatetimeArray>
['2026-02-02 00:00:00+00:00']
Length: 1, dtype: datetime64[ns, UTC]
valid: 1 unique values
  Values: <BooleanArray>
[True]
Length: 1, dtype: boolean
target_id: 4 unique values
  Values: <IntegerArray>
[0, 50, 100, 150]
Length: 4, dtype: Int64
classification_system: 18 unique values
  Values: <StringArray>
[                    'region',                       'side',
                       'flow',                'super_class',
                 'cell_class',                  'cell_type',
                'hemilineage',         'sexually_dimorphic',
      'neuropeptide_verified', 'neurotransmitter_predicted',
         'fafb_783_cell_type',         'manc_121_cell_type',
    'hemibrain_121_cell_type',          'fafb_783_match_id',
          'manc_121_match_id',     'hemibrain_121_match_id',
                    'user_id',  'neurotransmitter_verified']
Length: 18, dtype: string
cell_type: 29 unique values
  Values: <StringA

## Query neurons by hemilineage + neuropil/side
Set parameters below, then run to find matching neurons.

In [39]:
# --- Set query parameters ---
hemilineage = '04B'
side = 'right'
neuropil = 't2'  # Options: 't1', 't2', 't3' (mapped to 'soma in T1', etc. in cell_info)

# --- Query codex_annotations ---
# codex_annotations uses a long-format key-value schema:
#   classification_system = key (e.g. 'hemilineage', 'side', 'region')
#   cell_type = value (e.g. '12B', 'right', 'ventral_nerve_cord')
# Each neuron has multiple rows, one per classification_system.
# We query each filter separately, then intersect by pt_root_id.

# 1) Get all neurons with the target hemilineage
hl_df = client.materialize.query_table('codex_annotations',
    filter_equal_dict={'classification_system': 'hemilineage', 'cell_type': hemilineage})
print(f'Neurons with hemilineage={hemilineage}: {hl_df.shape[0]}')

# 2) Get all neurons on the target side
side_df = client.materialize.query_table('codex_annotations',
    filter_equal_dict={'classification_system': 'side', 'cell_type': side})
print(f'Neurons on {side} side: {side_df.shape[0]}')

# 3) Get all neurons in VNC (codex region level)
region_df = client.materialize.query_table('codex_annotations',
    filter_equal_dict={'classification_system': 'region', 'cell_type': 'ventral_nerve_cord'})
print(f'Neurons in VNC: {region_df.shape[0]}')

# 4) Get MANC match IDs
manc_id = client.materialize.query_table('codex_annotations',
    filter_equal_dict={'classification_system': 'manc_121_match_id'})
print(f'Neurons with MANC id: {manc_id.shape[0]}')

# 5) Get MANC cell types
manc_type = client.materialize.query_table('codex_annotations',
    filter_equal_dict={'classification_system': 'manc_121_cell_type'})
print(f'Neurons with MANC type: {manc_type.shape[0]}')

Neurons with hemilineage=04B: 476
Neurons on right side: 66873
Neurons in VNC: 21299
Neurons with MANC id: 22582
Neurons with MANC type: 22488


In [40]:
manc_type.head(40)

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position
0,979120,2026-02-02 00:00:00+00:00,True,126,manc_121_cell_type,DNxn153,126,2025-07-16 00:22:25.259763+00:00,True,76070400102424848,720575941412619631,"[470784, 217072, 166140]"
1,978995,2026-02-02 00:00:00+00:00,True,0,manc_121_cell_type,DNp32,0,2025-07-16 00:22:25.259763+00:00,True,75366094051243980,720575941472733451,"[392556, 146948, 143955]"
2,978996,2026-02-02 00:00:00+00:00,True,1,manc_121_cell_type,DNd02,1,2025-07-16 00:22:25.259763+00:00,True,76212442992686915,720575941662395704,"[487572, 370000, 125100]"
3,978997,2026-02-02 00:00:00+00:00,True,2,manc_121_cell_type,DNg100,2,2025-07-16 00:22:25.259763+00:00,True,76211068401754593,720575941626500746,"[486292, 207380, 88155]"
4,978998,2026-02-02 00:00:00+00:00,True,3,manc_121_cell_type,AN07B004,3,2025-07-16 00:22:25.259763+00:00,True,76636785895693328,720575941631721954,"[536312, 629532, 146340]"
5,978999,2026-02-02 00:00:00+00:00,True,4,manc_121_cell_type,AN07B004,4,2025-07-16 00:22:25.259763+00:00,True,76566417218149633,720575941566084871,"[527628, 627804, 150615]"
6,979000,2026-02-02 00:00:00+00:00,True,5,manc_121_cell_type,DNg100,5,2025-07-16 00:22:25.259763+00:00,True,76985399465468463,720575941500851362,"[575588, 238348, 86175]"
7,979001,2026-02-02 00:00:00+00:00,True,6,manc_121_cell_type,DNd03,6,2025-07-16 00:22:25.259763+00:00,True,76353180480628309,720575941620273535,"[499900, 370000, 125775]"
8,979002,2026-02-02 00:00:00+00:00,True,7,manc_121_cell_type,DNp30,7,2025-07-16 00:22:25.259763+00:00,True,76212443059856089,720575941393242628,"[486364, 370000, 137610]"
9,979003,2026-02-02 00:00:00+00:00,True,8,manc_121_cell_type,AN19B001,8,2025-07-16 00:22:25.259763+00:00,True,76353180547663916,720575941508609665,"[501592, 370000, 128295]"


In [41]:
# # 3) Get all neurons in VNC (codex region level)
# pt_root_id_df = client.materialize.query_table('codex_annotations',
#     filter_equal_dict={'pt_root_id': 720575941532368106})
# # print(f'Neurons in VNC: {region_df.shape[0]}')


In [42]:
# Intersect by pt_root_id: hemilineage ∩ side ∩ region(VNC)
hl_side = hl_df[hl_df['pt_root_id'].isin(side_df['pt_root_id'])]
# all VNC hl neurons
hl_side = hl_df.copy()

bundle_neurons = hl_side[hl_side['pt_root_id'].isin(region_df['pt_root_id'])].copy()
print(f'Hemilineage={hemilineage}, side={side}, VNC: {bundle_neurons.shape[0]} neurons')

# Optional: further filter by neuropil (T1/T2/T3) using cell_info table
# cell_info uses tag2='soma in VNC', tag='soma in T1'/'soma in T2'/'soma in T3'
neuropil_tag = f'soma in {neuropil.upper()}'  # e.g. 'soma in T2'
try:
    np_df = client.materialize.query_table('cell_info',
        filter_equal_dict={'tag2': 'soma in VNC', 'tag': neuropil_tag})
    np_filtered = bundle_neurons[bundle_neurons['pt_root_id'].isin(np_df['pt_root_id'])]
    if len(np_filtered) > 0:
        bundle_neurons = np_filtered.copy()
        print(f'After neuropil filter ({neuropil_tag}): {bundle_neurons.shape[0]} neurons')
    else:
        print(f'No overlap with cell_info {neuropil_tag} annotations — using full VNC set')
except Exception as e:
    print(f'cell_info neuropil query failed ({e}) — using full VNC set')

# Merge MANC cell type into bundle_neurons
manc_type_lookup = manc_type[['pt_root_id', 'cell_type']].rename(columns={'cell_type': 'manc_type'})
bundle_neurons = bundle_neurons.merge(manc_type_lookup, on='pt_root_id', how='left')
print(f'\nNeurons with MANC type: {bundle_neurons["manc_type"].notna().sum()} / {bundle_neurons.shape[0]}')

print(f'\nFinal bundle_neurons: {bundle_neurons.shape[0]} neurons')
bundle_neurons.head()

Hemilineage=04B, side=right, VNC: 456 neurons
No overlap with cell_info soma in T2 annotations — using full VNC set

Neurons with MANC type: 423 / 456

Final bundle_neurons: 456 neurons


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pt_supervoxel_id,pt_root_id,pt_position,manc_type
0,657748,2026-02-02 00:00:00+00:00,True,5193,hemilineage,04B,5193,2025-07-16 00:22:25.259763+00:00,True,76918260604001711,720575941478849857,"[567764, 627124, 100710]",IN04B066
1,682122,2026-02-02 00:00:00+00:00,True,75981,hemilineage,04B,75981,2025-07-16 00:22:25.259763+00:00,True,76004566844000871,720575941482712506,"[459904, 755724, 170460]",IN04B108
2,682133,2026-02-02 00:00:00+00:00,True,76005,hemilineage,04B,76005,2025-07-16 00:22:25.259763+00:00,True,75582010713958951,720575941731770923,"[410492, 718492, 151515]",IN04B027
3,684314,2026-02-02 00:00:00+00:00,True,90376,hemilineage,04B,90376,2025-07-16 00:22:25.259763+00:00,True,77270104861663437,720575941652547601,"[607948, 630696, 191070]",IN04B015
4,684316,2026-02-02 00:00:00+00:00,True,90378,hemilineage,04B,90378,2025-07-16 00:22:25.259763+00:00,True,75862798563287885,720575941566467847,"[445104, 632324, 167310]",IN04B034


In [43]:
VOXEL_NM = client.info.viewer_resolution()
VOXEL_NM
def nm2pxl_xform(pt_position):
    """
    Convert [x_nm, y_nm, z_nm] -> [x_px, y_px, z_px] using voxel size 4x4x45 nm.
    Returns floats (sub-pixel allowed). Use rounding/casting if you want integer indices.
    """
    if pt_position is None:
        return None
    if len(pt_position) != 3:
        raise ValueError(f"Expected 3 elements [x,y,z] in nm, got {len(pt_position)}: {pt_position}")

    x_nm, y_nm, z_nm = pt_position
    sx, sy, sz = VOXEL_NM
    return [x_nm / sx, y_nm / sy, z_nm / sz]

def nm2pxl_xform_int(pt_position,method: str = "round"):
    """
    Same conversion, but returns integer voxel indices.
    method: 'round' | 'floor' | 'ceil' | 'trunc'
    """
    import math

    x, y, z = nm2pxl_xform(pt_position)

    if method == "round":
        f = lambda v: int(round(v))
    elif method == "floor":
        f = lambda v: int(math.floor(v))
    elif method == "ceil":
        f = lambda v: int(math.ceil(v))
    elif method == "trunc":
        f = lambda v: int(v)
    else:
        raise ValueError("method must be one of: round, floor, ceil, trunc")

    return [f(x), f(y), f(z)]


bundle_neurons['pt_position_nm'] = bundle_neurons['pt_position']
bundle_neurons['pt_position'] = bundle_neurons['pt_position'].apply(nm2pxl_xform_int)


## Generate Neuroglancer state
Build a state showing the known neurons of this hemilineage/region so you can visually identify additional neurons in the same bundle.

In [44]:
SPELUNKER_SITE = "https://spelunker.cave-explorer.org/"

# Build base state
seg_source = client.info.segmentation_source().replace("graphene://", "graphene://middleauth+")
state = (
    ViewerState(dimensions=list(client.info.viewer_resolution()), infer_coordinates=False)
    .add_image_layer(source=client.info.image_source(), name='banc')
    .add_segmentation_layer(source=seg_source, name='seg')
    .add_segmentation_layer(
        source='gs://lee-lab_brain-and-nerve-cord-fly-connectome/region_outlines/|neuroglancer-precomputed:',
        name='reg',
        segments=[1, 3],
        segment_colors={1: '#2A7FFF', 3: '#D343D6'},
        alpha_3d=0.05,
    )
)

layer_name = f'{hemilineage}_{side}_{neuropil}'

plot_df = bundle_neurons.copy()
if not plot_df.empty:
    # If loaded from CSV, pt_position will be strings and needs parsing
    if plot_df['pt_position'].dtype == 'object' and isinstance(plot_df['pt_position'].iloc[0], str):
        plot_df['pt_position'] = split_pt_pos_str_lists(plot_df)

    # Sort: untyped (NaN manc_type) first so they appear at top of annotation list
    plot_df = plot_df.sort_values('manc_type', na_position='first')
    plot_df['manc_type'] = plot_df['manc_type'].fillna('untyped')

    state.add_points(
        data=plot_df,
        name=layer_name,
        point_column='pt_position',
        description_column='manc_type',
        linked_segmentation='seg',
        color='#FF6600',
    )
else:
    print('No neurons found — generating base state only')

# Patch seg layer directly: '!id' = starred but mesh hidden
state_dict = state.to_dict()
if not plot_df.empty:
    seg_ids = plot_df['pt_root_id'].tolist()
    for layer in state_dict.get('layers', []):
        if layer.get('name') == 'seg':
            layer['segments'] = [f'!{seg_id}' for seg_id in seg_ids]
            break

state_id = client.state.upload_state_json(state_dict)
url = client.state.build_neuroglancer_url(state_id, ngl_url=SPELUNKER_SITE).replace("/?json_url=", "#!middleauth+")
print('Open this URL to view and annotate:')
print(url)

Open this URL to view and annotate:
https://spelunker.cave-explorer.org/#!middleauth+https://global.daf-apis.com/nglstate/api/v1/6534913800011776


## Import new annotations from Neuroglancer state
After annotating new neurons in Neuroglancer, paste the state URL below to import them.

In [ ]:
# Paste your Neuroglancer state URL here
new_state_url = ''  # <-- paste URL

import re
def _strip_state_int(url):
    match = re.search(r'/(\d+)$', url)
    if match:
        return int(match.group(1))
    return None

if new_state_url:
    state_int = _strip_state_int(new_state_url)
    new_jst = client.state.get_state_json(state_int)
    
    state_anno_df = parser.annotation_dataframe(new_jst)
    state_layers = parser.annotation_layers(new_jst)
    state_anno_df['point'] = state_anno_df['point'].apply(lambda x: [int(i) for i in x])
    
    print(f'Annotation layers: {state_layers}')
    print(f'Total annotations: {state_anno_df.shape[0]}')
    state_anno_df.head()
else:
    print('Paste a Neuroglancer state URL above and re-run this cell.')

## Review & save
Combine known and newly annotated neurons, review, and save to CSV.

In [ ]:
import os

# Combine known + new annotations
if new_state_url and not state_anno_df.empty:
    # Rename columns to match bundle table schema
    new_annots = state_anno_df.rename(columns={'point': 'pt_position'}).copy()
    new_annots['classification_system'] = hemilineage
    new_annots['cell_type'] = '_'  # user can refine later
    
    combined = pd.concat([bundle_neurons, new_annots], ignore_index=True)
    print(f'Known: {bundle_neurons.shape[0]}, New: {new_annots.shape[0]}, Combined: {combined.shape[0]}')
    display(combined)
    
    # Save
    save_dir = './BANC_bundle_csvs'
    os.makedirs(save_dir, exist_ok=True)
    save_path = f'{save_dir}/{hemilineage}_{side}_{neuropil}_bundle.csv'
    combined.to_csv(save_path, index=False)
    print(f'Saved to {save_path}')
else:
    print('No new annotations to combine. Run the import cell above first.')

## Update CAVE table (once created)
After manually creating the BANC neurite bundle table in CAVE, uncomment and run the cell below to upload annotations.

In [ ]:
# # Once the BANC bundle table has been created:
# table_name = 'banc_neurite_bundle_cell_type_table_v0'  # adjust to actual table name
# A = annot8or.Annot8or(table_name=table_name, dataset=dataset, client=client)
#
# # Add the new annotations
# A.add_annotations(new_annots)
#
# # Upload — use dry_run=True first to verify
# A.upload_annotations(dry_run=True)
# # A.upload_annotations(dry_run=False)  # uncomment when ready
#
# # View the result
# A.get_state()